# Nemotron 3.5 Super VL: three finance LoRA adapters

This is the single entry-point notebook for three isolated [NeMo AutoModel](https://docs.nvidia.com/nemo/automodel/latest/recipes-e2e-examples/sft-peft)  LoRA runs:

1. **Visual OCR** — [TAT-DQA](https://huggingface.co/datasets/next-tat/TAT-DQA) span and multi-span image questions.
2. **Reasoning** — TAT-DQA arithmetic and count over released text evidence.
3. **Code generation** — CodeFinQA and CodeTAT-QA Python programs form [BizBench](https://huggingface.co/datasets/kensho/bizbench)

This notebook guides you through preparing the finance datasets, configuring NeMo Automodel, and training three LoRA adapters for visual OCR, reasoning, and code generation. It then serves the adapters through [NVIDIA Dynamo](#start-dynamo-load-adapters-and-prepare-routing) and evaluates them on the matching test tasks. 

Run the cells in order to see data checks, training progress, generated examples, and evaluation plots directly in the notebook. The workflow saves the trained adapters, predictions, and evaluation metrics for any future use. 

Model: `nvidia/Nemotron-3.5-Super-VL-120B-A12B-GA-candidate-MTP-boosted-20261001`. Use this original BF16 snapshot without conversion or quantized substitution. MTP-head training remains disabled.

Run `bash setup.sh` on your GPU machine first; it handles asset downloads, environment setup and local Jupyter startup. Then select **NeMo AutoModel** and run each code cell in order. Save this same notebook to retain its outputs.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, yaml

def discover_workspace():
    override = os.environ.get("NEMO_LORA_WORKSPACE")
    if override:
        return Path(override).expanduser().resolve()
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        flat = (candidate / "training.py").is_file()
        packaged = (candidate / "scripts" / "training.py").is_file()
        if flat or packaged:
            return candidate
    raise RuntimeError("Set NEMO_LORA_WORKSPACE or launch this notebook from the project tree")

WORKSPACE = discover_workspace()
SCRIPT_ROOT = WORKSPACE / "scripts" if (WORKSPACE / "scripts" / "data.py").is_file() else WORKSPACE
for import_root in (SCRIPT_ROOT, WORKSPACE):
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))
AUTOMODEL_ROOT = Path(os.environ.get("AUTOMODEL_ROOT", WORKSPACE / "Automodel")).expanduser().resolve()
from model import MODEL_ID, MODEL_REVISION, MODEL_LOCAL_DIR, inspect_snapshot, require_runtime_support
PYTHON = AUTOMODEL_ROOT / ".venv" / "bin" / "python"
HF_HOME = WORKSPACE / "hf-cache"
MODEL_SNAPSHOT = Path(os.environ.get("NEMO_LORA_MODEL_PATH", WORKSPACE / MODEL_LOCAL_DIR)).expanduser().absolute()
TAT_SOURCE_DIR = Path(os.environ.get("NEMO_LORA_TAT_SOURCE_DIR", WORKSPACE / "data" / "sources" / "tat-dqa")).expanduser().resolve()
BIZ_SOURCE_DIR = Path(os.environ.get("NEMO_LORA_BIZ_SOURCE_DIR", WORKSPACE / "data" / "sources" / "bizbench")).expanduser().resolve()
CORE_DATASET = WORKSPACE / "data" / "finance_core_tatdqa_bizbench_v2"
# Two hardware profiles: H100 (SM90) and GB200 (SM100). Auto-select from CUDA.
from environment import PROFILES, detect_profile
from environment import verify as verify_automodel_source
GPU_PROFILE = "auto"  # Keep auto; an explicit H100/GB200 choice must match detected GPUs.
HARDWARE = detect_profile(source="torch", requested=GPU_PROFILE)
GPU_COUNT = HARDWARE["num_gpus"]
GPU_CONFIG = WORKSPACE / HARDWARE["config_file"]
assert GPU_CONFIG.is_file(), f"Missing GPU-specific AutoModel YAML: {GPU_CONFIG}"
print(f"Selected AutoModel GPU configuration: {GPU_CONFIG}")
print(json.dumps({"available_profiles": list(PROFILES), "selected_hardware": HARDWARE}, indent=2))
host_profile = WORKSPACE / "logs" / "hardware-host.json"
if host_profile.is_file():
    earlier = json.loads(host_profile.read_text())
    assert (earlier["profile"], earlier["num_gpus"]) == (HARDWARE["profile"], GPU_COUNT), "GPU visibility changed since host setup"
os.environ["TORCH_CUDA_ARCH_LIST"] = HARDWARE["torch_cuda_arch_list"]
AUTOMODEL_SOURCE = verify_automodel_source(WORKSPACE, AUTOMODEL_ROOT)
print(json.dumps({"automodel_source": AUTOMODEL_SOURCE}, indent=2))

TASKS = ("visual_ocr", "reasoning", "code_generation")
TASK_RUNNER = SCRIPT_ROOT / "training.py"
assert TASK_RUNNER.is_file(), f"Missing training entry point: {TASK_RUNNER}"
from notebook import run_visible, require_adapters, training_guard
from notebook import request_start
# Always evaluate the complete trace-free test sets.

os.environ["HF_HOME"] = str(HF_HOME)
os.environ["PYTHONPATH"] = os.pathsep.join((str(SCRIPT_ROOT), str(WORKSPACE), os.environ.get("PYTHONPATH", "")))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_OFFLINE", "1")
assert Path(sys.prefix).resolve() == PYTHON.parent.parent.resolve(), "Select the NeMo AutoModel kernel before running cells"
assert PYTHON.is_file(), f"Missing AutoModel Python: {PYTHON}"
assert (MODEL_SNAPSHOT / "config.json").is_file(), f"Download the pinned model snapshot first: {MODEL_SNAPSHOT}"
MODEL_AUDIT = inspect_snapshot(MODEL_SNAPSHOT)
print(json.dumps(MODEL_AUDIT, indent=2))
print({"workspace": str(WORKSPACE), "model_snapshot": str(MODEL_SNAPSHOT), "tat_source": str(TAT_SOURCE_DIR), "bizbench_source": str(BIZ_SOURCE_DIR), "visible_gpus": GPU_COUNT})

## Prepare datasets

Download both datasets from Hugging Face before running the notebook. This cell prepares the task datasets and checks that the splits do not overlap and test examples contain no reasoning traces.

In [ ]:
from datasets import load_from_disk
from data import TASK_DATASETS, task_dataset_paths

if not (CORE_DATASET / "split_manifest.json").is_file():
    run_visible([
        str(PYTHON), str(SCRIPT_ROOT / "data.py"),
        "--tat-source-dir", str(TAT_SOURCE_DIR),
        "--biz-source-dir", str(BIZ_SOURCE_DIR),
        "--output-dir", str(CORE_DATASET),
        "--seed", "42", "--validation-ratio", "0.1", "--render-dpi", "150",
    ])

run_visible([str(PYTHON), str(SCRIPT_ROOT / "data.py"),
             "--task-views-from", str(CORE_DATASET)])
TASK_PATHS = task_dataset_paths(WORKSPACE / "data")
seen_ids = set()
summary = {}
for capability, spec in TASK_DATASETS.items():
    path = TASK_PATHS[capability]
    manifest = json.loads((path / "split_manifest.json").read_text())
    assert manifest["capability"] == capability
    assert set(manifest["tasks"]) == set(spec["tasks"])
    assert all(value == 0 for pair in manifest["overlaps"].values() for value in pair.values())
    dataset = load_from_disk(str(path))
    ids = set()
    for split in ("train", "validation"):
        assert all(str(value).strip() for value in dataset[split]["reasoning"])
        assert all(str(value).startswith("<think>\n") for value in dataset[split]["target"])
        ids.update(dataset[split]["sample_id"])
    test = dataset["test"]
    assert all(not str(value).strip() for value in test["reasoning"])
    assert all(not str(value).strip() for value in test["program"])
    assert all(str(target).strip() == str(answer).strip() for target, answer in zip(test["target"], test["answer"]))
    ids.update(test["sample_id"])
    assert not (ids & seen_ids)
    seen_ids.update(ids)
    summary[capability] = manifest["counts"]
print({"datasets": summary, "total_unique_rows": len(seen_ids), "checkpoint_format": MODEL_AUDIT["checkpoint_format"], "test_reference_traces": 0})

## Generate each task's AutoModel YAML

This cell creates a training configuration for each adapter using the Nemo Automodel recipes and the detected GPU type (H100 or GB200). It sets the model and dataset paths and checks the BF16 checkpoint. Each adapter gets a one-epoch training budget, and the setup is verified when training starts. 

In [ ]:
for capability in TASKS:
    run_visible([
        str(PYTHON), str(TASK_RUNNER), "task", "--capability", capability, "--workspace", str(WORKSPACE),
        "--automodel-root", str(AUTOMODEL_ROOT), "--model-path", str(MODEL_SNAPSHOT),
        "--num-gpus", str(GPU_COUNT), "--stage", "config",
        "--gpu-profile", HARDWARE["profile"],
    ])

CONFIGS = {name: WORKSPACE / "configs" / f"finance_{name}_lora_bf16_20261001.yaml" for name in TASKS}
for capability, path in CONFIGS.items():
    cfg = yaml.safe_load(path.read_text())
    assert cfg["dataset"]["capability"] == capability
    assert cfg["validation_dataset"]["capability"] == capability
    assert cfg["optimizer"]["lr"] == 1e-4
    assert cfg["distributed"]["ep_size"] == GPU_COUNT
    assert cfg["model"]["num_nextn_predict_layers"] == 0
    print(capability, {"config": str(path), "steps": cfg["step_scheduler"]["max_steps"], "checkpoint": cfg["checkpoint"]["checkpoint_dir"]})

## Train the three adapters

This cell uses NeMo Automodel to train three separate LoRA adapters:
- **Visual OCR**
- **Reasoning**
- **Code-generation**

LoRA keeps the BF16 base model frozen and trains smaller task-specific updates. Separate adapters keep these updates independent. This helps address Catastrophic forgetting, where learning a new task weakens previously learned capabitlities: training the reasoning adapter does not overwrite the OCR or code-generation adapters. However, each adapter can still change the base model's behavior while active, so performance must be checked through evaluation. 

Each adapter uses a one-epoch training budget, with regular validation and checkpoint saves. During inference, the workflow selects the task's adapter and sends the request through NVIDIA Dynamo, which supports serving multiple adapters over a shared base model.

In [ ]:
require_runtime_support("training")
with training_guard(WORKSPACE):
    for capability in TASKS:
        print(f"TRAIN: {capability}", flush=True)
        run_visible([
            str(PYTHON), str(TASK_RUNNER), "task", "--capability", capability, "--workspace", str(WORKSPACE),
            "--automodel-root", str(AUTOMODEL_ROOT), "--model-path", str(MODEL_SNAPSHOT),
            "--num-gpus", str(GPU_COUNT), "--stage", "train",
            "--gpu-profile", HARDWARE["profile"],
        ])
ADAPTERS = require_adapters(WORKSPACE)

## Start Dynamo, load adapters, and prepare routing

[NVIDIA Dynamo](https://docs.nvidia.com/dynamo/) is an inference-serving framework that manages model workers and routes request to them. Here, it works with vLLM to [support the three LoRA adapters](https://docs.nvidia.com/dynamo/dev/cli/operations/lo-ra-adapters) over a shared BF16 base model, avoiding a separate full model copy for each task. Adapters can be loaded without restarting the serving worker. 

The following cell connects training to inference:
1. Checks the training completed successfully for all the three adapters.
2. Starts the workspace's Dynamo serving container through the helper created during [setup](setup.sh). Jupyter does not need direct Docker access. 
3. Waits for the serving worker, loads the adapters, and confirms they are available. 
4. Prepares the shuffled task-based test requests for task-based routing. 

The workflow selects selects an adapter using each sample's task tag: `visual OCR`, `code generation`, or `reasoning`. It send that adapter's name to Dynamo, which routes the request to the serving worker. This is explicit routing, not a a learned classifier.

Continue to inference only after all three adapters have loaded successfully. If startup fails, check `logs/dynamo-vllm-worker.log`

In [ ]:
require_runtime_support("serving")
ADAPTERS = require_adapters(WORKSPACE)
request_start(WORKSPACE)
run_visible([
    str(PYTHON), str(WORKSPACE / "dynamo" / "sitecustomize.py"),
    "--workspace", str(WORKSPACE), "--timeout", "1800",
])
run_visible([
    str(PYTHON), str(SCRIPT_ROOT / "evaluation.py"), "route",
    "--workspace", str(WORKSPACE), "--model-id", MODEL_ID,
    "--model-revision", MODEL_REVISION, "--seed", "42",
    "--output-name", "finance_three_adapter_routed_test_bf16_20261001", "--plan-only",
])
print("Three adapters loaded; shuffled route plan is ready.")

## Run the full shuffled test evaluation

This cell sends the shuffled task-specific adapters through Dynamo and saves the generated responses as they are processed. Results are reported by task adapter. TAT-DQA questions include supporting document crops or text. Generated Python code is checked for syntax but is not executed. That is not the scope of this notebook. 

In [ ]:
require_adapters(WORKSPACE)
run_visible([
    str(PYTHON), str(SCRIPT_ROOT / "evaluation.py"), "evaluate",
    "--workspace", str(WORKSPACE), "--seed", "42",
    "--output-name", "finance_three_adapter_routed_test_bf16_20261001", "--resume",
])
ROUTED_DIR = WORKSPACE / "evaluation" / "finance_three_adapter_routed_test_bf16_20261001"
report = json.loads((ROUTED_DIR / "routed_metrics.json").read_text())
assert report["coverage"] == 1.0, "Evaluation incomplete; inspect logs and rerun this cell"
assert report["evaluated_samples"] == report["expected_samples"]
print(json.dumps(report, indent=2))

## Evaluated responses

After evaluation, this cell displays one saved model response from each adapater. It shows generated response only. 

In [ ]:
ROUTED_DIR = WORKSPACE / "evaluation" / "finance_three_adapter_routed_test_bf16_20261001"
predictions_path = ROUTED_DIR / "routed_predictions.jsonl"
if predictions_path.is_file():
    rows = [json.loads(line) for line in predictions_path.read_text().splitlines() if line.strip()]
    shown = set()
    for row in rows:
        if row["capability"] in shown:
            continue
        shown.add(row["capability"])
        print("=" * 80)
        print(f"ADAPTER: {row['capability']} | TASK: {row['task']}")
        print(f"QUESTION:\n{row['question']}")
        print(f"MODEL RESPONSE:\n{row['raw_completion']}")
else:
    print("No routed predictions yet; run the inference cell first")

## Evaluation result plots

This cell summarizes the saved inference results in four plots:
- **Performance by adapter**: compares the task accuracy, exact match, relaxed match, and answer F1 across visual OCR, reasoning, and code generation. 
- **Performance by task**: shows which tasks perform well and which need improvement. OCR scoring includes text overlap; numerical answers allow a 5% tolerance. 
- **Samples by task**: shows how many examples support each score. Results based on very few samples should be interpreted cautiously. 
- **Evaluations and output checks**: shows evaluation coverage, reasoning-output formatting, and Python syntax validity. 

Run this cell only after the inference completes. 

In [ ]:
metrics_path = ROUTED_DIR / "routed_metrics.json"
if metrics_path.is_file():
    import matplotlib.pyplot as plt
    report = json.loads(metrics_path.read_text())
    capability_metrics = report["by_capability"]
    task_metrics = report["by_task"]
    fig, axes = plt.subplots(2, 2, figsize=(16, 11))
    names = list(capability_metrics)
    metric_names = ("task_accuracy", "exact_accuracy", "relaxed_accuracy", "answer_f1")
    width = 0.2
    x = list(range(len(names)))
    for offset, metric in enumerate(metric_names):
        axes[0, 0].bar(
            [value + (offset - 1.5) * width for value in x],
            [capability_metrics[name][metric] for name in names],
            width=width,
            label=metric.replace("_", " "),
        )
    axes[0, 0].set_xticks(x, names)
    axes[0, 0].set(title="Metrics by routed adapter", ylabel="Score", ylim=(0, 1))
    axes[0, 0].legend(fontsize=8)
    task_names = list(task_metrics)
    axes[0, 1].barh(task_names, [task_metrics[name]["task_accuracy"] for name in task_names], label="task accuracy")
    axes[0, 1].barh(task_names, [task_metrics[name]["answer_f1"] for name in task_names], alpha=0.45, label="answer F1")
    axes[0, 1].set(title="Metrics by held-out task", xlabel="Score", xlim=(0, 1))
    axes[0, 1].legend(fontsize=8)
    axes[1, 0].bar(task_names, [task_metrics[name]["samples"] for name in task_names])
    axes[1, 0].set(title="Evaluated samples by task", ylabel="Samples")
    axes[1, 0].tick_params(axis="x", rotation=30)
    diagnostic_names = ["coverage", "closed trace", "Python syntax"]
    diagnostic_values = [report["coverage"], report["closed_trace_rate"], report["python_syntax_rate"]]
    axes[1, 1].bar(diagnostic_names, diagnostic_values)
    axes[1, 1].set(title="Coverage and generation diagnostics", ylabel="Rate", ylim=(0, 1))
    plt.tight_layout()
    plt.show()
    print({
        "evaluated_samples": report["evaluated_samples"],
        "coverage": report["coverage"],
        "task_accuracy": report["task_accuracy"],
        "exact_accuracy": report["exact_accuracy"],
        "relaxed_accuracy": report["relaxed_accuracy"],
        "answer_f1": report["answer_f1"],
        "closed_trace_rate": report["closed_trace_rate"],
        "python_syntax_rate": report["python_syntax_rate"],
        "hit_max_new_tokens": report["hit_max_new_tokens"],
        "route_backend": report["route_backend"],
    })
else:
    print(f"Evaluation results pending: {metrics_path}")